# Local Linear Attention State Diagnostics

Compare causal and non-causal `pfns.model.linear_attention.LinearAttention` models by replaying the linear prefix state `S_t = sum_i k_i v_i^T`. The plots are intentionally minimal: associative residual, state norm, and drift from a reference pretraining-range state.

In [ ]:
from __future__ import annotations

import gc
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PFNs").exists():
    repo_root = repo_root.parent
if not (repo_root / "PFNs").exists():
    raise RuntimeError("Could not find repo root containing PFNs/.")
sys.path.insert(0, str(repo_root / "PFNs"))

from pfns.experiments.model_benchmarks.model_registry import MODEL_FAMILIES
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.model.linear_attention import LinearAttention

warnings.filterwarnings("ignore", message="ShortConvolution is crucial.*", category=UserWarning)
plt.rcParams["figure.dpi"] = 120

## Configuration

In [ ]:
MODEL_NAMES = [
    "Linear_Attention_Non_Causal",
    "Linear_Attention_Comb_ST",
]
MODEL_LABELS = {
    "Linear_Attention_Non_Causal": "LA non-causal",
    "Linear_Attention_Comb_ST": "LA causal",
    "Linear_Attention_Non_Causal_fro_norm": "LA non-causal fro",
    "Linear_Attention_Comb_ST_fro_norm": "LA causal fro",
}
LAYER_INDICES = [0, 6, 11]
TRAIN_CONTEXT_LEN = 128_000
N_TEST = 64
BATCH_SIZE = 1
NUM_FEATURES = 20
MAX_POINTS = 4096
POSITION_SAMPLING = "log"  # "log" or "linear"
REFERENCE_POSITION = 1_000
SMOOTH_WINDOW = 31
EPS = 1e-30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AUTOCAST_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
USE_AUTOCAST = DEVICE.type == "cuda"
print(f"device={DEVICE}, use_autocast={USE_AUTOCAST}, autocast_dtype={AUTOCAST_DTYPE}")

## Helpers

In [ ]:
def label(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name)


def find_config(model_name: str) -> dict:
    for registry in MODEL_FAMILIES.values():
        if model_name in registry:
            return registry[model_name]
    raise KeyError(f"Unknown model: {model_name}")


def load_model(model_name: str):
    models, configs = load_models_for_benchmark({model_name: model_configs[model_name]}, device=str(DEVICE))
    models[model_name].eval()
    return models[model_name], configs[model_name]


def linear_attention_layers(model) -> list[LinearAttention]:
    return [m for m in model.modules() if isinstance(m, LinearAttention)]


def sample_positions(seq_len: int) -> set[int]:
    if POSITION_SAMPLING == "linear":
        points = np.linspace(0, seq_len - 1, min(seq_len, MAX_POINTS)).round().astype(int)
    elif POSITION_SAMPLING == "log":
        points = np.r_[
            np.arange(min(seq_len, 64, MAX_POINTS), dtype=int),
            np.geomspace(1, seq_len, MAX_POINTS * 4).round().astype(int) - 1,
        ]
    else:
        raise ValueError(f"Unknown POSITION_SAMPLING={POSITION_SAMPLING!r}")
    required = {0, min(REFERENCE_POSITION - 1, seq_len - 1), seq_len - 1}
    points = sorted({int(p) for p in points if 0 <= p < seq_len} | required)
    if len(points) <= MAX_POINTS:
        return set(points)
    keep = set(points[:64]) | required
    rest = [p for p in points if p not in keep]
    need = MAX_POINTS - len(keep)
    rest = [rest[i] for i in np.linspace(0, len(rest) - 1, need).round().astype(int)] if need > 0 else []
    return keep | set(rest)


def flatten_seq_heads(x: torch.Tensor) -> torch.Tensor:
    return x.movedim(1, 0).reshape(x.shape[1], -1, x.shape[-1]).contiguous()


def capture_linear_attention_streams(model, model_name: str, x: torch.Tensor, y: torch.Tensor) -> dict[int, dict]:
    layers = linear_attention_layers(model)
    records = {}
    originals = {}

    for layer_index in LAYER_INDICES:
        layer = layers[layer_index]
        assert layer.state_update_rule == "linear", "This simple notebook supports only state_update_rule='linear'."
        originals[layer_index] = layer.incontext_fit

        def wrapped(layer_input, *, _idx=layer_index, _layer=layer, _orig=layer.incontext_fit):
            q, k, v = _layer._project_qkv(layer_input)
            _q, k = _layer._apply_query_key_feature_maps(q, k)
            records[_idx] = {
                "model_name": model_name,
                "model_label": label(model_name),
                "layer_index": _idx,
                "causal": bool(_layer.causal or _layer.causal_train_only),
                "use_k_sum_normalization": bool(_layer.use_k_sum_normalization),
                "k": flatten_seq_heads(k).detach().float().cpu(),
                "v": flatten_seq_heads(v).detach().float().cpu(),
            }
            return _orig(layer_input)

        layer.incontext_fit = wrapped

    try:
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_AUTOCAST):
            _ = model.incontext_fit(x, y)
    finally:
        for layer_index, original in originals.items():
            layers[layer_index].incontext_fit = original

    assert set(records) == set(LAYER_INDICES), f"Captured layers {sorted(records)}, expected {LAYER_INDICES}."
    return records


def linear_state_curve(record: dict) -> pd.DataFrame:
    k = record["k"].to(DEVICE)
    v = record["v"].to(DEVICE)
    seq_len, obs, key_dim = k.shape
    state = torch.zeros(obs, key_dim, v.shape[-1], device=DEVICE)
    k_sum = torch.zeros(obs, key_dim, device=DEVICE)
    ref_state = None
    positions = sample_positions(seq_len)
    rows = []

    for t in range(seq_len):
        prediction = torch.einsum("ok,okv->ov", k[t], state)
        avg_prediction = prediction / max(t, 1)
        if record["use_k_sum_normalization"]:
            denom = torch.einsum("ok,ok->o", k[t], k_sum).unsqueeze(-1)
            prediction = prediction / (denom + EPS)
            avg_prediction = avg_prediction / (denom / max(t, 1) + EPS)

        state += k[t].unsqueeze(-1) * v[t].unsqueeze(-2)
        k_sum += k[t]
        if t == min(REFERENCE_POSITION - 1, seq_len - 1):
            ref_state = state.detach().clone()

        if t in positions:
            residual_norm = (v[t] - prediction).norm(dim=-1).clamp_min(EPS)
            avg_residual_norm = (v[t] - avg_prediction).norm(dim=-1).clamp_min(EPS)
            value_norm = v[t].norm(dim=-1).clamp_min(EPS)
            state_norm = torch.linalg.matrix_norm(state, ord="fro", dim=(-2, -1)).clamp_min(EPS)
            update_norm = k[t].norm(dim=-1) * value_norm
            drift = torch.zeros_like(state_norm) if ref_state is None else torch.linalg.matrix_norm(state - ref_state, ord="fro", dim=(-2, -1)) / torch.linalg.matrix_norm(ref_state, ord="fro", dim=(-2, -1)).clamp_min(EPS)
            rows.append({
                "model_name": record["model_name"],
                "model_label": record["model_label"],
                "layer_index": record["layer_index"],
                "causal": record["causal"],
                "position": t,
                "relative_residual": float((residual_norm / value_norm).mean()),
                "avg_relative_residual": float((avg_residual_norm / value_norm).mean()),
                "state_norm": float(state_norm.mean()),
                "update_norm": float(update_norm.mean()),
                "drift_from_ref": float(drift.mean()),
            })

    del k, v, state, k_sum, ref_state
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

## Run Diagnostics

In [ ]:
model_configs = {name: find_config(name) for name in MODEL_NAMES}
base_model, base_config = load_model(MODEL_NAMES[0])
batch = base_config.priors[0].create_get_batch_method()(
    batch_size=BATCH_SIZE,
    seq_len=TRAIN_CONTEXT_LEN + N_TEST,
    num_features=NUM_FEATURES,
    single_eval_pos=TRAIN_CONTEXT_LEN,
    n_targets_per_input=1,
)
train_x = batch.x[:, :TRAIN_CONTEXT_LEN].to(DEVICE)
train_y = batch.y[:, :TRAIN_CONTEXT_LEN].to(DEVICE)
print(f"train_x={tuple(train_x.shape)}, train_y={tuple(train_y.shape)}")

curves = []
for i, model_name in enumerate(MODEL_NAMES):
    model = base_model if i == 0 else load_model(model_name)[0]
    try:
        records = capture_linear_attention_streams(model, model_name, train_x, train_y)
        for layer_index, record in records.items():
            print(f"{label(model_name)} layer {layer_index}: k={tuple(record['k'].shape)}, v={tuple(record['v'].shape)}, causal={record['causal']}")
            curves.append(linear_state_curve(record))
    finally:
        if i != 0:
            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

la_df = pd.concat(curves, ignore_index=True)
display(la_df.head())

## Plots

In [ ]:
COLORS = {name: f"C{i}" for i, name in enumerate(MODEL_NAMES)}
METRICS = {
    "relative_residual": "mean ||S_{t-1} k_t - v_t|| / ||v_t||",
    "avg_relative_residual": "mean ||(S_{t-1}/t) k_t - v_t|| / ||v_t||",
    "state_norm": "mean ||S_t||_F",
    "drift_from_ref": f"mean ||S_t - S_{REFERENCE_POSITION}||_F / ||S_{REFERENCE_POSITION}||_F",
    "update_norm": "mean ||k_t v_t^T||_F",
}


def smooth(y: np.ndarray, window: int = SMOOTH_WINDOW) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if window <= 1 or y.size < 3:
        return y
    window = min(int(window) | 1, y.size if y.size % 2 else y.size - 1)
    return np.convolve(np.pad(y, window // 2, mode="edge"), np.ones(window) / window, mode="valid")


def plot_metric(metric: str, *, yscale: str = "log") -> None:
    for layer_index in LAYER_INDICES:
        fig, ax = plt.subplots(figsize=(10, 3.5))
        for model_name in MODEL_NAMES:
            curve = la_df[(la_df.model_name == model_name) & (la_df.layer_index == layer_index)].sort_values("position")
            ax.plot(curve.position + 1, smooth(curve[metric]), label=label(model_name), color=COLORS[model_name], linewidth=1.8)
        ax.axvline(REFERENCE_POSITION, color="black", linewidth=0.8, alpha=0.35)
        ax.set_xscale("log")
        ax.set_yscale(yscale)
        ax.set_xlabel(f"sequence position ({POSITION_SAMPLING}-sampled)")
        ax.set_ylabel(METRICS[metric])
        ax.set_title(f"{METRICS[metric]} | layer {layer_index}")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()


for metric in METRICS:
    plot_metric(metric)

## Summary

In [ ]:
def nearest_rows(position: int) -> pd.DataFrame:
    return pd.DataFrame([curve.loc[(curve.position - position).abs().idxmin()] for _, curve in la_df.groupby(["model_name", "layer_index"])])

near_ref = nearest_rows(REFERENCE_POSITION).set_index(["model_name", "layer_index"])
near_end = nearest_rows(TRAIN_CONTEXT_LEN - 1).set_index(["model_name", "layer_index"])
summary = near_end[["model_label", "position", "relative_residual", "avg_relative_residual", "state_norm", "drift_from_ref", "update_norm"]].rename(columns={"position": "final_position"})
summary["ref_relative_residual"] = near_ref.relative_residual
summary["ref_avg_relative_residual"] = near_ref.avg_relative_residual
summary["final_over_ref_residual"] = summary.relative_residual / summary.ref_relative_residual.clip(lower=EPS)
summary["final_over_ref_avg_residual"] = summary.avg_relative_residual / summary.ref_avg_relative_residual.clip(lower=EPS)
display(summary.reset_index().sort_values(["layer_index", "model_name"]))